In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain.tools import  tool
from typing import Dict,Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase
tavily_client=TavilyClient()
db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web using Tavily."""
    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:
    """Execute an SQL query on the Chinook database."""
    try:
        return db.run(query)
    except Exception as e:
        return f"error: {e}"

In [4]:
from dataclasses import dataclass
@dataclass
class UserRole:
    user_role:str='external'

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dymanic_tool_call(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:

    if request.runtime.context.user_role != "internal":
        request = request.override(tools=[web_search])

    return handler(request)

In [6]:
from langchain.agents import create_agent
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[web_search, sql_query],
    middleware=[dymanic_tool_call],
    context_schema=UserRole,
)

In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="How many assistants are in the database?")
        ]
    },
    context=UserRole(user_role="internal")
)

print(response["messages"][-1].content)

The Chinook database does not have a table named "assistants" or "employees". The Chinook database is a sample database that contains information about music, including artists, albums, tracks, and invoices. It does not contain information about assistants or employees. If you have any other questions or need help with something else, I'll be happy to assist you.
